# 01 — Phase 1: PDF Download + YOLO Crop Extraction
Downloads each PDF directly into RAM, runs DocLayout-YOLO, saves figure crops.
PDFs are **not saved to disk** — only the crops are written.

**Before running:** check config.yaml → subset.mode and n_first.

In [1]:
# ── CELL 1: Setup ────────────────────────────────────────────────
import sys; sys.path.insert(0, "..")
from src.config_loader import load_config
from src.patents import load_patents, get_subset
from src.extractor import extract_crops_streaming

cfg = load_config()

print("Config loaded.")
print(f"  Subset mode : {cfg['subset']['mode']}")
print(f"  n_first     : {cfg['subset']['n_first']}")
print(f"  YOLO device : {cfg['extractor']['device']}")
print(f"  PDF DPI     : {cfg['extractor']['pdf_dpi']}")
print(f"  Crops saved : {cfg['paths']['crops']}")

Config loaded.
  Subset mode : all
  n_first     : 5
  YOLO device : cuda:1
  PDF DPI     : 150
  Crops saved : /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Drive_files_to_syncronize/3 - Images Processing & DataSets/2-Images_DataSets/1627/1627_Patents_Raw_Crops


---
**Optional:** If you ran  and saved a list, load it here instead of using the config subset mode.

In [2]:
# ── CELL 2: Load patents ──────────────────────────────────────────
# Option A: use config.yaml subset settings (default)
df, missing_df = load_patents(cfg)   # missing_df = patents with no PDF URL
subset         = get_subset(df, cfg)

# Option B: load from saved filter list (uncomment to use)
# from pathlib import Path
# ids = Path("../selected_patents.txt").read_text().strip().splitlines()
# subset = df[df["Record Number"].isin(ids)].copy()
# print(f"Loaded {len(subset)} patents from saved filter")

print(f"Ready to process {len(subset)} patents.")
subset[["Record Number", "Title"]].head(5)

Loading Excel: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Drive_files_to_syncronize/2 - Patente & Validation/3 -Raw_Patent_Exports_PatSeer_&Gold_Standard/1627__dataset_26_05_26.xlsx


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  11 patents have no PDF URL (will appear in status report)
  Loaded 1616 patents with valid PDF URLs.
  Subset mode='all': 1616 patents selected.
Ready to process 1616 patents.


,Record Number,Title
0,US2022267016A1,ELECTRICALLY POWERED ROTARY-WING AIRCRAFT
1,US11787551B1,Vertical takeoff and landing aircraft electric...
2,US2022234745A1,AIR VEHICLE CONFIGURATIONS
3,US2020148347A1,ELECTRIC TILTROTOR AIRCRAFT
4,US2015014475A1,Vertical Takeoff and Landing (VTOL) Air Vehicle


In [ ]:
# ── CELL 3: Download + Extract (streaming — no PDFs saved) ──────────
# Downloads each PDF into RAM, runs YOLO, saves crops. No PDF files written.
# Already-processed patents are  skipped  automatically.
# missing_df passed so no-URL patents also appear in extraction_status.xlsx.

crop_results = extract_crops_streaming(subset, cfg, no_url_df=missing_df)

# Quick summary
total_crops = sum(len(v) for v in crop_results.values())
empty       = [k for k, v in crop_results.items() if len(v) == 0]
print(f"Total crops: {total_crops}")
print(f"Patents with 0 crops: {len(empty)}")
if empty:
    print("  ", empty[:10])

Download + Extract:  20%|██        | 331/1616 [02:14<09:02,  2.37it/s]


  FAILED to download GB202410221D0: 504 Server Error: Gateway Time-out for url: https://api.patseer.com/data/pdf/IBa9A%24%24%24okAbumPDisHI%2BCEOPMu5kRH%24%24%24esrQjkGU7dMz11EZw0W3tRkUW%24%24%24nqisWuclvhwfnU2VV2%24%24%24yAbId%2BqFQcfMZealccq%24%24%244KdrmbJf7XiiD1LOWdB5oA4e1%2BDqvX6PJwGlCVVBSvd%2BuSq%2BPh3Ais9lm4f1U5GtN0gfP%24%24%243js9H8%3D


Download + Extract:  39%|███▊      | 626/1616 [05:18<08:52,  1.86it/s]


  FAILED to download GB201801059D0: 504 Server Error: Gateway Time-out for url: https://api.patseer.com/data/pdf/IBa9A%24%24%24okAbumPDisHI%2BCEOPMu5kRH%24%24%24esidF1tSlxN6Z1EZw0W3tRkUW%24%24%24nqisWuclvhwfnU2VV2%24%24%24yAbId%2BqFQcecJqTVuUuaEKdrmbJf7XiiD1LOWdB5oA4e1%2BDqvX6PJwGlCVVBSvd%2BuSq%2BPh3Ais9lm4f1U5GtN0gfP%24%24%243js9H8%3D


Download + Extract:  44%|████▍     | 718/1616 [11:07<19:53,  1.33s/it]


  FAILED to download GB202006780D0: 504 Server Error: Gateway Time-out for url: https://api.patseer.com/data/pdf/IBa9A%24%24%24okAbumPDisHI%2BCEOPMu5kRH%24%24%24estjLvkIcS8k11EZw0W3tRkUW%24%24%24nqisWuclvhwfnU2VV2%24%24%24yAbId%2BqFQcXK7%2BsqkgMxRKdrmbJf7XiiD1LOWdB5oA4e1%2BDqvX6PJwGlCVVBSvd%2BuSq%2BPh3Ais9lm4f1U5GtN0gfP%24%24%243js9H8%3D


Download + Extract:  69%|██████▊   | 1108/1616 [16:09<09:41,  1.14s/it]


  FAILED to download ITMI20001822A0: 504 Server Error: Gateway Time-out for url: https://api.patseer.com/data/pdf/IBa9A%24%24%24okAbumPDisHI%2BCEOPMu5kRH%24%24%24esP39XzyTny%2BB1EZw0W3tRkUW%24%24%24nqisWuclvhwfnU2VV2%24%24%24S4tn1Ferdng6akk%2B0TaVLIIkr0cvn7INTXZCoaJ%24%24%24FvjwPkhXW7ly57DZratSFyoKQpcl3lG0GXX07UqnFAecGZ7Hv3Iu34Uc%3D


Download + Extract: 100%|██████████| 1616/1616 [18:30<00:00,  1.46it/s]

Status report: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Drive_files_to_syncronize/3 - Images Processing & DataSets/2-Images_DataSets/1627/1627_Patents_logs/extraction_status.xlsx
Extraction complete: 24113 crops saved to /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Drive_files_to_syncronize/3 - Images Processing & DataSets/2-Images_DataSets/1627/1627_Patents_Raw_Crops
Total crops: 24113
Patents with 0 crops: 6
   ['GB202410221D0', 'GB201801059D0', 'GB202006780D0', 'US2011048438A1', 'GB201307770D0', 'ITMI20001822A0']


In [4]:
# ── CELL 4: Save crop index for Phase 2 ─────────────────────────────
# Saves a JSON map of { record_id: [list of crop file paths] }
# Phase 2 (02_processing.ipynb) reads this file to know what to process.

import json
from pathlib import Path

crop_index  = {k: [str(p) for p in v] for k, v in crop_results.items()}
out         = Path(cfg["paths"]["logs"]) / "crop_index.json"
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w") as f:
    json.dump(crop_index, f, indent=2)
print(f"Crop index saved to: {out}")

Crop index saved to: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Drive_files_to_syncronize/3 - Images Processing & DataSets/2-Images_DataSets/1627/1627_Patents_logs/crop_index.json
